# Two-Stage PSA System Analysis


---

## Quick Start Guide

**Welcome!** This notebook analyzes a two-stage Pressure Swing Adsorption (PSA) system for hydrogen purification.

### How to Use This Notebook

1. **Run All Cells**: Click **Runtime** > **Run all** (or press `Ctrl+F9`) to execute the entire notebook
2. **Adjust Parameters**: Use the sliders in the "Interactive Parameters" section to change operating conditions
3. **View Results**: Results update automatically when you change parameters

### What You'll Learn
- H2 recovery and purity calculations
- Mass balance across all streams
- Sensitivity analysis for recycle fractions
- O2 safety analysis for flammability concerns

---

In [ ]:
# @title 1. Setup - Run This Cell First { display-mode: "form" }
# @markdown **Click the play button** or press `Shift+Enter` to initialize the model.
# @markdown
# @markdown This cell loads all required libraries and the PSA calculation model.


from dataclasses import dataclass, field
from typing import TypedDict

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from numpy.typing import NDArray

print("Loading libraries...")
# Set plot style
plt.style.use(
    "seaborn-v0_8-whitegrid"
    if "seaborn-v0_8-whitegrid" in plt.style.available
    else "seaborn-whitegrid"
)
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 11


class ComponentData(TypedDict):
    name: str
    feed_pct: float
    stage1_removal_pct: float
    stage2_removal_pct: float


DEFAULT_COMPONENTS: list[ComponentData] = [
    {
        "name": "H2",
        "feed_pct": 32.08,
        "stage1_removal_pct": 18.0,
        "stage2_removal_pct": 15.0,
    },
    {
        "name": "CO",
        "feed_pct": 38.22,
        "stage1_removal_pct": 98.0,
        "stage2_removal_pct": 99.99,
    },
    {
        "name": "CO2",
        "feed_pct": 21.98,
        "stage1_removal_pct": 98.0,
        "stage2_removal_pct": 99.99,
    },
    {
        "name": "H2O",
        "feed_pct": 4.85,
        "stage1_removal_pct": 99.0,
        "stage2_removal_pct": 99.99,
    },
    {
        "name": "N2",
        "feed_pct": 0.50,
        "stage1_removal_pct": 95.0,
        "stage2_removal_pct": 99.99,
    },
    {
        "name": "O2",
        "feed_pct": 0.50,
        "stage1_removal_pct": 81.0,
        "stage2_removal_pct": 99.99,
    },
    {
        "name": "CH4",
        "feed_pct": 1.88,
        "stage1_removal_pct": 99.0,
        "stage2_removal_pct": 99.99,
    },
]


@dataclass
class StreamFlows:
    fresh_feed: NDArray[np.float64]
    s2_tail_recycle: NDArray[np.float64]
    product_recycle: NDArray[np.float64]
    mixed_feed: NDArray[np.float64]
    exhaust: NDArray[np.float64]
    s2_tail_vent: NDArray[np.float64]
    interstage: NDArray[np.float64]
    gross_product: NDArray[np.float64]
    s2_tail: NDArray[np.float64]
    net_product: NDArray[np.float64]


@dataclass
class StreamCompositions:
    fresh_feed: NDArray[np.float64]
    s2_tail_recycle: NDArray[np.float64]
    product_recycle: NDArray[np.float64]
    mixed_feed: NDArray[np.float64]
    exhaust: NDArray[np.float64]
    s2_tail_vent: NDArray[np.float64]
    interstage: NDArray[np.float64]
    gross_product: NDArray[np.float64]
    s2_tail: NDArray[np.float64]
    net_product: NDArray[np.float64]


@dataclass
class PSAResults:
    component_names: list[str]
    flows: StreamFlows
    compositions: StreamCompositions
    h2_recovery_pct: float
    h2_purity_pct: float
    total_feed_scfm: float
    total_net_product_scfm: float
    total_exhaust_scfm: float
    total_s2_tail_vent_scfm: float
    mass_balance_error: float
    s2_tail_h2_pct: float
    s2_tail_o2_pct: float


@dataclass
class PSAModel:
    total_feed_scfm: float = 1100.0
    s2_tail_recycle_frac: float = 1.0
    product_recycle_frac: float = 0.0
    components: list[ComponentData] = field(
        default_factory=lambda: list(DEFAULT_COMPONENTS)
    )

    def calculate(self) -> PSAResults:
        n_components = len(self.components)
        component_names = [c["name"] for c in self.components]
        feed_pct = np.array([c["feed_pct"] for c in self.components], dtype=np.float64)
        r1 = np.array(
            [c["stage1_removal_pct"] / 100.0 for c in self.components], dtype=np.float64
        )
        r2 = np.array(
            [c["stage2_removal_pct"] / 100.0 for c in self.components], dtype=np.float64
        )
        r_tail = self.s2_tail_recycle_frac
        r_prod = self.product_recycle_frac

        fresh_feed = self.total_feed_scfm * feed_pct / np.sum(feed_pct)
        denominator = 1.0 - (1.0 - r1) * (r2 * r_tail + (1.0 - r2) * r_prod)
        mixed_feed = fresh_feed / denominator
        exhaust = mixed_feed * r1
        interstage = mixed_feed - exhaust
        s2_tail = interstage * r2
        s2_tail_recycle = s2_tail * r_tail
        s2_tail_vent = s2_tail * (1.0 - r_tail)
        gross_product = interstage - s2_tail
        product_recycle = gross_product * r_prod
        net_product = gross_product * (1.0 - r_prod)

        flows = StreamFlows(
            fresh_feed=fresh_feed,
            s2_tail_recycle=s2_tail_recycle,
            product_recycle=product_recycle,
            mixed_feed=mixed_feed,
            exhaust=exhaust,
            s2_tail_vent=s2_tail_vent,
            interstage=interstage,
            gross_product=gross_product,
            s2_tail=s2_tail,
            net_product=net_product,
        )

        def calc_composition(flow_array):
            total = np.sum(flow_array)
            if total == 0:
                return np.zeros(n_components, dtype=np.float64)
            return flow_array / total * 100.0

        compositions = StreamCompositions(
            fresh_feed=calc_composition(fresh_feed),
            s2_tail_recycle=calc_composition(s2_tail_recycle),
            product_recycle=calc_composition(product_recycle),
            mixed_feed=calc_composition(mixed_feed),
            exhaust=calc_composition(exhaust),
            s2_tail_vent=calc_composition(s2_tail_vent),
            interstage=calc_composition(interstage),
            gross_product=calc_composition(gross_product),
            s2_tail=calc_composition(s2_tail),
            net_product=calc_composition(net_product),
        )

        h2_idx = component_names.index("H2")
        o2_idx = component_names.index("O2")

        return PSAResults(
            component_names=component_names,
            flows=flows,
            compositions=compositions,
            h2_recovery_pct=float(net_product[h2_idx] / fresh_feed[h2_idx] * 100.0),
            h2_purity_pct=float(compositions.net_product[h2_idx]),
            total_feed_scfm=self.total_feed_scfm,
            total_net_product_scfm=float(np.sum(net_product)),
            total_exhaust_scfm=float(np.sum(exhaust)),
            total_s2_tail_vent_scfm=float(np.sum(s2_tail_vent)),
            mass_balance_error=float(
                np.sum(fresh_feed)
                - np.sum(exhaust)
                - np.sum(s2_tail_vent)
                - np.sum(net_product)
            ),
            s2_tail_h2_pct=float(compositions.s2_tail[h2_idx]),
            s2_tail_o2_pct=float(compositions.s2_tail[o2_idx]),
        )


def get_flammability_status(h2_pct, o2_pct):
    if o2_pct < 0.1:
        return "Safe-Low O2", "green"
    if h2_pct > 4 and o2_pct > 2:
        return "CRITICAL", "red"
    if h2_pct < 4:
        return "Safe-Below LFL", "green"
    if h2_pct > 75:
        return "Caution-Rich", "orange"
    return "FLAMMABLE", "red"


print("\n" + "=" * 50)
print("  PSA Model loaded successfully!")
print("=" * 50)
print("\nProceed to the next cell to configure parameters.")

---

## Process Flow Diagram

```
                                    Product recycle
                              ┌────────────────────────────────────┐      3R
                              │                                    │
                              │                          Gross product    Net product
                              │                               3G              3N
    Feed      Mixed feed      ▼         Interstage       ┌────────┴────────────►
      1    5A          5B    5C             6             │
    ──────►●────────────►[Comp]────►[PSA 1]────────────►[PSA 2]
           │                           │                  │
           │                           │    2             │
           │                           └──────►           │
           │                             Exhaust          │
           │         4                                    │
           └──────────────────────────────────────────────┘
                         Recycle (S2 Tail)
```

---

In [ ]:
# @title 2. Interactive Calculator { display-mode: "form", run: "auto" }
# @markdown ## Operating Parameters
# @markdown Adjust these sliders to see how they affect PSA performance.
# @markdown **Results update automatically!**

# @markdown ---
# @markdown ### Flow Rates
total_feed_scfm = 1100  # @param {type:"slider", min:500, max:2000, step:50}

# @markdown ### Recycle Fractions
s2_tail_recycle_pct = 100  # @param {type:"slider", min:0, max:100, step:5}
product_recycle_pct = 0  # @param {type:"slider", min:0, max:50, step:5}

# @markdown ---
# @markdown ### Feed Composition
feed_h2_pct = 32.08  # @param {type:"number"}
feed_o2_pct = 0.5  # @param {type:"number"}

# @markdown ---
# @markdown ### Stage 1 Removal Efficiencies
s1_h2_removal = 18  # @param {type:"slider", min:0, max:50, step:1}
s1_o2_removal = 81  # @param {type:"slider", min:50, max:99, step:1}

# Build components
user_components = [
    {
        "name": "H2",
        "feed_pct": feed_h2_pct,
        "stage1_removal_pct": float(s1_h2_removal),
        "stage2_removal_pct": 15.0,
    },
    {
        "name": "CO",
        "feed_pct": 38.22,
        "stage1_removal_pct": 98.0,
        "stage2_removal_pct": 99.99,
    },
    {
        "name": "CO2",
        "feed_pct": 21.98,
        "stage1_removal_pct": 98.0,
        "stage2_removal_pct": 99.99,
    },
    {
        "name": "H2O",
        "feed_pct": 4.85,
        "stage1_removal_pct": 99.0,
        "stage2_removal_pct": 99.99,
    },
    {
        "name": "N2",
        "feed_pct": 0.50,
        "stage1_removal_pct": 95.0,
        "stage2_removal_pct": 99.99,
    },
    {
        "name": "O2",
        "feed_pct": feed_o2_pct,
        "stage1_removal_pct": float(s1_o2_removal),
        "stage2_removal_pct": 99.99,
    },
    {
        "name": "CH4",
        "feed_pct": 1.88,
        "stage1_removal_pct": 99.0,
        "stage2_removal_pct": 99.99,
    },
]

# Calculate
model = PSAModel(
    total_feed_scfm=float(total_feed_scfm),
    s2_tail_recycle_frac=s2_tail_recycle_pct / 100.0,
    product_recycle_frac=product_recycle_pct / 100.0,
    components=user_components,
)
results = model.calculate()
status, color = get_flammability_status(results.s2_tail_h2_pct, results.s2_tail_o2_pct)

# Display results
print("\n" + "=" * 60)
print("                    KEY RESULTS")
print("=" * 60)
print(f"\n  H2 Recovery:        {results.h2_recovery_pct:>10.2f} %")
print(f"  H2 Purity:          {results.h2_purity_pct:>10.5f} %")
print(f"  Net Product Flow:   {results.total_net_product_scfm:>10.2f} SCFM")
print(f"  Exhaust Flow:       {results.total_exhaust_scfm:>10.2f} SCFM")
print(f"\n  Mass Balance Error: {results.mass_balance_error:>10.2e}")

print("\n" + "-" * 60)
print("                    SAFETY STATUS")
print("-" * 60)
print(f"\n  S2 Tail H2:         {results.s2_tail_h2_pct:>10.2f} %")
print(f"  S2 Tail O2:         {results.s2_tail_o2_pct:>10.2f} %")
if color == "red":
    print(f"\n  ⚠️  STATUS: {status} - DANGER! O2 > 2% with H2 > 4%")
elif color == "orange":
    print(f"\n  ⚠️  STATUS: {status} - Monitor conditions")
else:
    print(f"\n  ✅ STATUS: {status}")
print("\n" + "=" * 60)

In [ ]:
# @title 3. View Mass Balance Table { display-mode: "form", run: "auto" }
# @markdown Click to display the detailed mass balance table.

mass_balance_data = {
    "Component": results.component_names,
    "Fresh Feed": results.flows.fresh_feed,
    "Mixed Feed": results.flows.mixed_feed,
    "Exhaust": results.flows.exhaust,
    "Interstage": results.flows.interstage,
    "Net Product": results.flows.net_product,
}

df_mass = pd.DataFrame(mass_balance_data)
totals = df_mass.sum(numeric_only=True)
totals["Component"] = "TOTAL"
df_mass = pd.concat([df_mass, pd.DataFrame([totals])], ignore_index=True)

print("\nMASS BALANCE TABLE (SCFM)")
print("=" * 80)
display(df_mass.style.format(precision=4).set_properties(**{"text-align": "right"}))

In [ ]:
# @title 4. Sensitivity Analysis Plots { display-mode: "form", run: "auto" }
# @markdown ### Plot Options
show_lines = True  # @param {type:"boolean"}
show_markers = False  # @param {type:"boolean"}
num_points = 51  # @param {type:"slider", min:11, max:101, step:10}

# Determine line style
if show_lines and show_markers:
    linestyle = "-"
    marker = "o"
    markersize = 4
elif show_lines:
    linestyle = "-"
    marker = None
    markersize = 0
elif show_markers:
    linestyle = "none"
    marker = "o"
    markersize = 6
else:
    linestyle = "-"
    marker = None
    markersize = 0

# Calculate sensitivity
s2_recycle_range = np.linspace(0, 1, num_points)
prod_recycle_range = np.array([0.0, 0.1, 0.2])

h2_recovery_data = {pr: [] for pr in prod_recycle_range}
net_product_data = {pr: [] for pr in prod_recycle_range}

for pr in prod_recycle_range:
    for s2r in s2_recycle_range:
        m = PSAModel(
            total_feed_scfm=float(total_feed_scfm),
            s2_tail_recycle_frac=float(s2r),
            product_recycle_frac=float(pr),
            components=user_components,
        )
        r = m.calculate()
        h2_recovery_data[pr].append(r.h2_recovery_pct)
        net_product_data[pr].append(r.total_net_product_scfm)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# H2 Recovery
ax1 = axes[0]
for pr in prod_recycle_range:
    ax1.plot(
        s2_recycle_range * 100,
        h2_recovery_data[pr],
        linestyle=linestyle,
        marker=marker,
        markersize=markersize,
        label=f"Product Recycle = {pr * 100:.0f}%",
    )
ax1.axvline(
    x=s2_tail_recycle_pct, color="red", linestyle="--", alpha=0.7, label="Current"
)
ax1.set_xlabel("Stage 2 Tail Recycle (%)")
ax1.set_ylabel("H2 Recovery (%)")
ax1.set_title("H2 Recovery vs Recycle Fractions")
ax1.legend()
ax1.grid(True, alpha=0.3)

# Net Product
ax2 = axes[1]
for pr in prod_recycle_range:
    ax2.plot(
        s2_recycle_range * 100,
        net_product_data[pr],
        linestyle=linestyle,
        marker=marker,
        markersize=markersize,
        label=f"Product Recycle = {pr * 100:.0f}%",
    )
ax2.axvline(
    x=s2_tail_recycle_pct, color="red", linestyle="--", alpha=0.7, label="Current"
)
ax2.set_xlabel("Stage 2 Tail Recycle (%)")
ax2.set_ylabel("Net Product Flow (SCFM)")
ax2.set_title("Net Product Flow vs Recycle Fractions")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# @title 5. O2 Safety Analysis { display-mode: "form", run: "auto" }
# @markdown This plot shows how S2 Tail O2 concentration varies with Stage 1 O2 removal efficiency.
# @markdown
# @markdown **Red zone indicates dangerous conditions (>2% O2 with high H2).**

# @markdown ### Plot Options
show_lines_o2 = True  # @param {type:"boolean"}
show_markers_o2 = False  # @param {type:"boolean"}

if show_lines_o2 and show_markers_o2:
    ls, mk, ms = "-", "o", 5
elif show_lines_o2:
    ls, mk, ms = "-", None, 0
elif show_markers_o2:
    ls, mk, ms = "none", "o", 6
else:
    ls, mk, ms = "-", None, 0

inlet_o2_values = [0.5, 1.0, 2.0, 5.0]
s1_removal_range = np.linspace(50, 99, 50)

fig, ax = plt.subplots(figsize=(12, 7))

colors = plt.cm.viridis(np.linspace(0, 0.9, len(inlet_o2_values)))

for j, inlet_o2 in enumerate(inlet_o2_values):
    o2_results = []
    for s1_rem in s1_removal_range:
        mod_components = [dict(c) for c in user_components]
        for c in mod_components:
            if c["name"] == "O2":
                c["feed_pct"] = inlet_o2
                c["stage1_removal_pct"] = float(s1_rem)
        m = PSAModel(float(total_feed_scfm), 1.0, 0.0, mod_components)
        o2_results.append(m.calculate().s2_tail_o2_pct)

    ax.plot(
        s1_removal_range,
        o2_results,
        linestyle=ls,
        marker=mk,
        markersize=ms,
        color=colors[j],
        linewidth=2,
        label=f"Inlet O2 = {inlet_o2}%",
    )

ax.axhline(
    y=2.0, color="red", linestyle="--", linewidth=2, label="DANGER THRESHOLD (2% O2)"
)
ax.fill_between(s1_removal_range, 2.0, 50, alpha=0.15, color="red")
ax.axvline(
    x=s1_o2_removal,
    color="green",
    linestyle=":",
    alpha=0.8,
    linewidth=2,
    label=f"Current ({s1_o2_removal}%)",
)

ax.set_xlabel("Stage 1 O2 Removal (%)", fontsize=12)
ax.set_ylabel("Stage 2 Tail O2 Concentration (%)", fontsize=12)
ax.set_title(
    "O2 Safety Analysis: S2 Tail O2% vs Stage 1 Removal Efficiency", fontsize=14
)
ax.legend(loc="upper right")
ax.grid(True, alpha=0.3)
ax.set_xlim([50, 99])
ax.set_ylim([0, max(10, ax.get_ylim()[1])])

plt.tight_layout()
plt.show()

In [ ]:
# @title 6. 3D Recovery Surface { display-mode: "form", run: "auto" }
# @markdown Interactive 3D plot showing H2 recovery as a function of both recycle fractions.


s2_range = np.linspace(0, 1, 25)
prod_range = np.linspace(0, 0.5, 15)
S2, PROD = np.meshgrid(s2_range, prod_range, indexing="ij")

recovery_surface = np.zeros_like(S2)
for i, s2r in enumerate(s2_range):
    for j, pr in enumerate(prod_range):
        m = PSAModel(float(total_feed_scfm), float(s2r), float(pr), user_components)
        recovery_surface[i, j] = m.calculate().h2_recovery_pct

fig = plt.figure(figsize=(12, 8))
ax = fig.add_subplot(111, projection="3d")
surf = ax.plot_surface(
    S2 * 100, PROD * 100, recovery_surface, cmap="viridis", alpha=0.8, edgecolor="none"
)
ax.set_xlabel("S2 Tail Recycle (%)")
ax.set_ylabel("Product Recycle (%)")
ax.set_zlabel("H2 Recovery (%)")
ax.set_title("H2 Recovery Surface")
fig.colorbar(surf, shrink=0.5, label="H2 Recovery (%)")

# Mark current operating point
ax.scatter(
    [s2_tail_recycle_pct],
    [product_recycle_pct],
    [results.h2_recovery_pct],
    color="red",
    s=100,
    marker="*",
    label="Current",
)
ax.legend()

plt.tight_layout()
plt.show()

---

## Mathematical Model Reference

### Key Equation (Algebraic Solution)

$$M_i = \frac{F_i}{1 - (1 - R_{1,i}) \left[ R_{2,i} \cdot r_{tail} + (1 - R_{2,i}) \cdot r_{prod} \right]}$$

Where:
- $F_i$ = Fresh feed flow of component $i$ (SCFM)
- $M_i$ = Mixed feed flow of component $i$ (SCFM)
- $R_{1,i}$ = Stage 1 removal fraction
- $R_{2,i}$ = Stage 2 removal fraction
- $r_{tail}$ = Stage 2 tail recycle fraction
- $r_{prod}$ = Product recycle fraction

### Performance Metrics

$$\text{H}_2 \text{ Recovery} = \frac{K_{H_2}}{F_{H_2}} \times 100\%$$

$$\text{H}_2 \text{ Purity} = \frac{K_{H_2}}{\sum_i K_i} \times 100\%$$

---

## Safety Guidelines

| Condition | Status | Action |
|-----------|--------|--------|
| O2 < 0.1% | Safe | Normal operation |
| O2 < 2%, H2 > 75% | Caution | Fuel-rich, monitor |
| O2 > 2%, H2 > 4% | **CRITICAL** | **Immediate action required** |

---

*Model validated against Excel reference - all values match within 1e-10 relative tolerance.*